In [4]:
import pandas as pd
import requests
import io

def load_gdrive_parquet_to_memory(file_id):
    url = "https://drive.google.com/uc?export=download"
    session = requests.Session()

    response = session.get(url, params={'id': file_id}, stream=True)

    token = None
    for key, value in response.cookies.items():
        if key.startswith('download_warning'):
            token = value
            break

    if token:
        params = {'id': file_id, 'confirm': token}
        response = session.get(url, params=params, stream=True)

    if response.status_code != 200:
        raise Exception(f"Failed to load file. Status Code: {response.status_code}")

    print("Download to RAM complete. Parsing Parquet...")

    return pd.read_parquet(io.BytesIO(response.content))


file_id = "15aUDDBI8eXrX-jgSNnP-18N9k2KrjeGc"
try:
    print("Fetching data from Google Drive directly to memory...")
    df = load_gdrive_parquet_to_memory(file_id)
    print(f"Success! DataFrame shape: {df.shape}")
    print(df.head())
except Exception as e:
    print("Error:", e)

Fetching data from Google Drive directly to memory...
Download to RAM complete. Parsing Parquet...
Error: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
